**In Colab version training model**

!pip install transformers datasets accelerate fiftyone

In [ ]:
import fiftyone as fo
from fiftyone.utils.huggingface import load_from_hub

LABEL_MAP = {
    "total.total_price": "TOTAL_AMOUNT",
    "date": "DATE",
    "store_info.name": "VENDOR",
}

def map_label(original_label):
    return LABEL_MAP.get(original_label, "O")

def preprocess_dataset():
    print("Loading dataset...")
    dataset = load_from_hub("Voxel51/consolidated_receipt_dataset")

    processed_data = []

    for sample in dataset:
        words = []
        labels = []

        for det in sample.detections.detections:
            words.append(det["text"])
            labels.append(map_label(det.label))

        processed_data.append({
            "words": words,
            "labels": labels
        })

    return processed_data

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

LABEL_LIST = ["O", "TOTAL_AMOUNT", "DATE", "VENDOR"]

label2id = {label: i for i, label in enumerate(LABEL_LIST)}
id2label = {i: label for label, i in label2id.items()}


def tokenize_and_align_labels(example, tokenizer):
    tokenized_inputs = tokenizer(
        example["words"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    word_ids = tokenized_inputs.word_ids(0)

    labels = []
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        else:
            labels.append(label2id[example["labels"][word_idx]])

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


def prepare_data():
    data = preprocess_dataset()  # ✅ directly call

    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

    tokenized_data = [
        tokenize_and_align_labels(sample, tokenizer)
        for sample in data[:200]
    ]

    return Dataset.from_list(tokenized_data)

In [ ]:
from transformers import BertForTokenClassification, TrainingArguments, Trainer
from google.colab import files
import shutil

def train_model():
    dataset = prepare_data()

    model = BertForTokenClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id
    )

    training_args = TrainingArguments(
        output_dir="/content/models/",
        per_device_train_batch_size=8,
        num_train_epochs=2,
        logging_steps=10,
        save_strategy="epoch",
        fp16=True
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset
    )

    trainer.train()
    print("\n\n📥Model Saving.........")
    model.save_pretrained("/content/models/ner-model")
    print("\n✅Saved the model...")


def download_model():
    shutil.make_archive("ner-model", 'zip', "/content/models/ner-model")
    files.download("ner-model.zip")
    print("\n\n✅Downloaded Model NER")

train_model()
download_model()